# 循环神经网络 ()

In [1]:
import torch
from torch.nn import functional

In [2]:
import json
from collections import Counter

## 词袋

In [3]:
data_file = "./datasets/caocao.json"

# 构建词袋
counter = Counter()

# 打开数据文件
with open(data_file, mode="r", encoding="utf-8") as fd:
    # 加载为字典
    all_poetry = json.load(fd)
    for poetry in all_poetry:
        contents = "".join(poetry['paragraphs'])
        for word in contents:
            counter.update(word)

vocab_size = len(counter.keys())
word2id = {word: idx for idx, word in enumerate(counter.keys())}
id2word = {idx: word for idx, word in enumerate(counter.keys())}

In [16]:
print("词袋大小：", vocab_size)

词袋大小： 982


## 数据集

In [4]:
all_data = []

with open(data_file, mode="r", encoding="utf-8") as fd:
    all_poetry = json.load(fd)
    for poetry in all_poetry:
        contents = "".join(poetry['paragraphs'])
        all_data.append(contents)
        
text_peotry = "".join(all_data)

In [5]:
# 将长数据转成编号
text_seq = [word2id[word] for word in text_peotry]

In [6]:
# 切分数据集
import random
import torch
def data_iter_random(corpus_indices, batch_size, num_steps):
    # 减1是因为输出的索引x是相应输入的索引y加1
    num_examples = (len(corpus_indices) - 1) // num_steps   # 假设数据长度30，可用样本数量 4
    epoch_size = num_examples // batch_size     # 批次数量 2
    example_indices = list(range(num_examples))   # 样本编号
    random.shuffle(example_indices) # 把样本顺序打乱。0,1,2,3打乱

    # 返回从pos开始的长为num_steps的序列
    def _data(pos):
        return corpus_indices[pos: pos + num_steps]
    
    for i in range(epoch_size):  # 循环一次返回一个批次
        # 每次读取batch_size个随机样本
        i = i * batch_size
        batch_indices = example_indices[i: i + batch_size]
        X = [_data(j * num_steps) for j in batch_indices]
        Y = [_data(j * num_steps + 1) for j in batch_indices]
        
        yield torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.long)

# RNN实现

In [7]:
class RNN:

    def __init__(self, vocab_size, dim_hidden=64):
        """
        定义需要训练的权重W_i, b_i, W_h, b_h    w_o, b_o
        params:
            vocal_size: 词袋大小
            dim_hidden: 词转向量后的维度
        """
        self.vocab_size = vocab_size
        self.dim_hidden = dim_hidden

        self.W_ih = torch.randn(size=(self.vocab_size, self.dim_hidden)) * 0.01
        self.W_hh = torch.randn(size=(self.dim_hidden, self.dim_hidden)) * 0.01
        self.W_oh = torch.randn(size=(self.dim_hidden, self.vocab_size)) * 0.01

        self.b_ih = torch.zeros(self.dim_hidden)
        self.b_hh = torch.zeros(self.dim_hidden)
        self.b_oh = torch.zeros(self.vocab_size)

        self.W_ih.requires_grad = True
        self.W_hh.requires_grad = True
        self.W_oh.requires_grad = True
        self.b_ih.requires_grad = True
        self.b_hh.requires_grad = True
        self.b_oh.requires_grad = True

    def __call__(self, tokens):
        """
        前向传导推理计算
        params:
            tokens: 词的编号数组, 维度[N, Seq_len]
        """
        H = torch.zeros(tokens.shape[-1], self.dim_hidden)

        inputs = functional.one_hot(tokens, self.vocab_size).to(torch.float32)

        outputs = []

        Hs = []
        for x in inputs:
            H = torch.matmul(x, self.W_ih) + self.b_ih + torch.matmul(H, self.W_hh) + self.b_hh
            H = torch.tanh(H)

            y = torch.matmul(H, self.W_oh) + self.b_oh
            outputs.append(y)
            Hs.append(H)
            
        return torch.cat(outputs, dim=0), torch.cat(Hs, dim=0)

In [11]:
dataloader = data_iter_random(text_seq, 4, 32)
model = RNN(vocab_size, 128)

## 训练

In [12]:
# 1. 超参数
epoch = 10000
lr = 0.05
# 2. 损失函数与优化器
loss_f = torch.nn.CrossEntropyLoss()

# 3. 循环更新
for e in range(epoch):
    total_loss = 0
    num = 0
    dataloader = data_iter_random(text_seq, 8, 32)
    for x, y in dataloader:
        # 3.1 推理
        outputs, H = model(x)
        # 3.2 误差
        Y = y.contiguous().view(-1)
        loss = loss_f(outputs, Y.long())
        # 3.3 求导
        loss.backward()
        # 3.4 更新
        with torch.no_grad():
            model.W_ih -= lr * model.W_ih.grad
            model.W_hh -= lr * model.W_hh.grad
            model.W_oh -= lr * model.W_oh.grad
            model.b_ih -= lr * model.b_ih.grad
            model.b_hh -= lr * model.b_hh.grad
            model.b_oh -= lr * model.b_oh.grad
        # 3.5 清零
            model.W_ih.grad.zero_()
            model.W_hh.grad.zero_()
            model.W_oh.grad.zero_()
            model.b_ih.grad.zero_()
            model.b_hh.grad.zero_()
            model.b_oh.grad.zero_()

            total_loss += loss.item()
            num += 1
    if e % 100 == 0:
        print(f"epoch:{e:05d}, loss:{total_loss / num:.6f}")
print("training is Done")


epoch:00000, loss:6.880503
epoch:00100, loss:5.855315
epoch:00200, loss:5.794153
epoch:00300, loss:5.716375
epoch:00400, loss:5.657967
epoch:00500, loss:5.562145
epoch:00600, loss:5.465055
epoch:00700, loss:5.390905
epoch:00800, loss:5.282286
epoch:00900, loss:5.207434
epoch:01000, loss:5.133943
epoch:01100, loss:5.048841
epoch:01200, loss:4.968621
epoch:01300, loss:4.881764
epoch:01400, loss:4.803048
epoch:01500, loss:4.703179
epoch:01600, loss:4.624709
epoch:01700, loss:4.511196
epoch:01800, loss:4.431349
epoch:01900, loss:4.310952
epoch:02000, loss:4.194394
epoch:02100, loss:4.091133
epoch:02200, loss:3.981907
epoch:02300, loss:3.883163
epoch:02400, loss:3.776398
epoch:02500, loss:3.675408
epoch:02600, loss:3.580051
epoch:02700, loss:3.472626
epoch:02800, loss:3.419839
epoch:02900, loss:3.315764
epoch:03000, loss:3.241074
epoch:03100, loss:3.156617
epoch:03200, loss:3.077050
epoch:03300, loss:2.995053
epoch:03400, loss:2.915133
epoch:03500, loss:2.845428
epoch:03600, loss:2.787237
e

## 推理

In [13]:
prefix = "对酒当歌"
tokens = [word2id[word] for word in prefix]

tokens = torch.tensor([tokens])

num_predict = 16 # 预测个数
predicts = [] # 预测内容
for i in range(num_predict):
    # 预测
    w_i = model.W_ih
    outputs, hidden = model(tokens)
    # 取最大下标
    predict = torch.argmax(outputs[-1], dim=0).item()
    predicts.append(id2word[predict])

    tokens = torch.tensor([predict])

text = "".join(predicts)

In [14]:
text

'以言志，何时更逝去去去去去去去去'